# Run the full helper based workflow

So far we've built each helper on its own. Here we wire them together behind a small `ipywidgets` UI — upload a CSV, pick a target column, then click through the four stages (EDA → Preprocessing → Modeling → Evaluation) one at a time.

## 1 — Setup

In [ ]:
%pip install -q google-genai pandas scikit-learn matplotlib python-dotenv ipywidgets

In [ ]:
import io
import os
import sys
import warnings
import contextlib
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from dotenv import load_dotenv
from google import genai
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score

sys.path.append(str(Path("../01-Exploratory-Data-Analysis-Helpers").resolve()))
sys.path.append(str(Path("../02-Preprocessing-Helpers").resolve()))
sys.path.append(str(Path("../03-Modeling-Helpers").resolve()))
sys.path.append(str(Path("../04-Evaluation-Helpers").resolve()))

from eda_summary_helper import eda_summary_helper
from eda_visual_helpers import eda_visual_helper
from preprocessing_pipeline import preprocessing_pipeline
from classification_helper import classification_helper
from classification_eval_helper import compute_classification_metrics, flag_anomalies

warnings.filterwarnings("ignore")
load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
pd.set_option("display.max_colwidth", None)


In [ ]:
# Helpers
def call_silent(fn, *args, **kwargs):
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        result = fn(*args, **kwargs)
    return result


def call_silent_collect_figs(fn, *args, **kwargs):
    """Run a plotting helper, capture any matplotlib figures it draws.

    `eda_visual_helper` calls plt.show() internally; in VS Code that gets
    replayed on every widget re-render, which is what caused duplicate EDA
    output. We suppress plt.show during the call, then return the Figure
    objects so we can append them to the Output widget explicitly.
    """
    buffer = io.StringIO()
    original_show = plt.show
    plt.show = lambda *a, **k: None
    try:
        with contextlib.redirect_stdout(buffer):
            fn(*args, **kwargs)
        figs = [plt.figure(n) for n in plt.get_fignums()]
    finally:
        plt.show = original_show
    return figs


def set_status(msg):
    status.value = f"<i>{msg}</i>"


def set_btn_running(btn):
    btn.button_style = "warning"
    btn.disabled = True


def set_btn_done(btn):
    btn.button_style = "success"
    btn.disabled = False


def set_btn_idle(btn):
    btn.button_style = "primary"


# Widgets and state for the UI. The user uploads a CSV, then steps through the stages of the pipeline by clicking the buttons. Each stage's output is captured in its own Output widget so it doesn't get lost when the next stage runs. The `state` dict is where the handlers stash artifacts that the later stages need to read (e.g. the preprocessed data and model results).
# UI
uploader = widgets.FileUpload(accept=".csv", multiple=False, description="Upload CSV")
target_dd = widgets.Dropdown(options=[], description="Target:", disabled=True)

btn_eda = widgets.Button(
    description="1. Run EDA", button_style="primary", disabled=True
)
btn_prep = widgets.Button(
    description="2. Preprocess", button_style="primary", disabled=True
)
btn_model = widgets.Button(
    description="3. Training", button_style="primary", disabled=True
)
btn_eval = widgets.Button(
    description="4. Evaluate", button_style="primary", disabled=True
)

status = widgets.HTML(value="<i>Upload a CSV to begin.</i>")

out_eda = widgets.Output()
out_prep = widgets.Output()
out_model = widgets.Output()
out_eval = widgets.Output()

accordion = widgets.Accordion(children=[out_eda, out_prep, out_model, out_eval])
accordion.set_title(0, "Stage 1 — EDA")
accordion.set_title(1, "Stage 2 — Preprocessing")
accordion.set_title(2, "Stage 3 — Training")
accordion.set_title(3, "Stage 4 — Evaluation")

display(
    widgets.VBox(
        [
            uploader,
            target_dd,
            widgets.HBox([btn_eda, btn_prep, btn_model, btn_eval]),
            status,
            accordion,
        ]
    )
)

state = {"df": None, "path": None, "prep": None, "model": None}


# Upload


def on_upload(change):
    if not uploader.value:
        return

    file_info = (
        list(uploader.value)[0]
        if isinstance(uploader.value, tuple)
        else uploader.value[0]
    )

    if isinstance(file_info, dict):
        name = file_info["name"]
        content = file_info["content"]
    else:
        name = list(uploader.value.keys())[0]
        content = uploader.value[name]["content"]

    df = pd.read_csv(io.BytesIO(content))
    tmp_path = Path("_uploaded.csv").resolve()
    df.to_csv(tmp_path, index=False)

    state.update({"df": df, "path": str(tmp_path), "prep": None, "model": None})

    target_dd.options = list(df.columns)
    target_dd.disabled = False

    # reset buttons
    for btn in [btn_eda, btn_prep, btn_model, btn_eval]:
        set_btn_idle(btn)
        btn.disabled = True

    btn_eda.disabled = False
    btn_prep.disabled = False

    for o in (out_eda, out_prep, out_model, out_eval):
        o.clear_output(wait=True)

    status.value = f"<b>{name}</b> loaded: {df.shape[0]} rows × {df.shape[1]} columns."


uploader._observers = {}
uploader.observe(on_upload, names="value")


# EDA


def on_eda(_):
    df = state["df"]
    target = target_dd.value
    if df is None or not target:
        return

    set_btn_running(btn_eda)
    set_status("Running EDA...")

    out_eda.clear_output(wait=True)
    plt.close("all")

    # Summary — append the result directly, no `with out_eda:` context.
    eda_result = call_silent(
        eda_summary_helper,
        question=f"Summarise dataset for classification predicting {target}",
        frame=df,
    )
    out_eda.append_display_data(eda_result)

    # Visual helpers — collect the Figures they draw and append each one
    # explicitly. Matches the pattern used by the other stages.
    for fig in call_silent_collect_figs(
        eda_visual_helper, question=f"Distribution of {target}", frame=df
    ):
        out_eda.append_display_data(fig)
    plt.close("all")

    for fig in call_silent_collect_figs(
        eda_visual_helper, question="Correlation matrix", frame=df
    ):
        out_eda.append_display_data(fig)
    plt.close("all")

    set_status("EDA finished")
    set_btn_done(btn_eda)


# Preprocessing


def on_prep(_):
    raw_path = state["path"]
    target = target_dd.value
    if raw_path is None or not target:
        return

    set_btn_running(btn_prep)
    set_status("Preprocessing data...")

    out_prep.clear_output(wait=True)

    result = call_silent(
        preprocessing_pipeline,
        raw_path=raw_path,
        target_col=target,
        task="classification",
    )

    X_train, X_test = result.X_train_enc, result.X_test_enc
    y_train, y_test = result.y_train, result.y_test

    state["prep"] = dict(X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test)

    X_train_df = pd.DataFrame(X_train)
    class_balance = pd.Series(y_train).value_counts(normalize=True).round(3).to_dict()

    # Write directly into the Output widget. Mixing print() and display()
    # inside `with out_prep:` can drop the DataFrame's HTML repr in VS Code.
    out_prep.append_stdout(
        f"Train: {X_train_df.shape}  |  Test: {pd.DataFrame(X_test).shape}\n"
    )
    out_prep.append_stdout(f"Class balance (train): {class_balance}\n\n")
    out_prep.append_stdout("First rows of the encoded training set:\n")
    out_prep.append_display_data(X_train_df.head(5))

    set_status("Preprocessing finished")
    set_btn_done(btn_prep)

    set_btn_idle(btn_model)
    btn_model.disabled = False


# Modeling


def on_model(_):
    prep = state["prep"]
    if prep is None:
        set_status("Run preprocessing first")
        return

    set_btn_running(btn_model)
    set_status("Training model...")

    out_model.clear_output(wait=True)

    results = call_silent(
        classification_helper,
        prep["X_train"],
        prep["X_test"],
        prep["y_train"],
        prep["y_test"],
    )

    state["model"] = results
    out_model.append_stdout(f"Model selected : {results['model_name']}\n")
    out_model.append_stdout(f"Init kwargs    : {results['init_kwargs']}\n")

    set_status("Model training finished")
    set_btn_done(btn_model)

    set_btn_idle(btn_eval)
    btn_eval.disabled = False


# Evaluation


def on_eval(_):
    prep = state["prep"]
    results = state["model"]

    if prep is None or results is None:
        set_status("Run preprocessing and modeling first")
        return

    set_btn_running(btn_eval)
    set_status("Evaluating model...")

    out_eval.clear_output(wait=True)

    metrics = call_silent(
        compute_classification_metrics,
        prep["y_test"],
        results["y_pred"],
        results["y_proba"],
    )

    out_eval.append_stdout("Metrics:\n")
    out_eval.append_display_data(pd.DataFrame([metrics]))

    # Sanity check: a dummy classifier that always predicts the majority
    # class. If our model isn't meaningfully better, metrics are misleading.
    dummy = DummyClassifier(strategy="most_frequent").fit(
        prep["X_train"], prep["y_train"]
    )
    baseline_df = pd.DataFrame(
        {
            "metric": ["accuracy", "f1 (weighted)"],
            "dummy": [
                round(dummy.score(prep["X_test"], prep["y_test"]), 3),
                round(
                    f1_score(
                        prep["y_test"],
                        dummy.predict(prep["X_test"]),
                        average="weighted",
                    ),
                    3,
                ),
            ],
            "our_model": [
                round(metrics["accuracy"], 3),
                round(
                    f1_score(prep["y_test"], results["y_pred"], average="weighted"),
                    3,
                ),
            ],
        }
    )
    out_eval.append_stdout("\nBaseline comparison (dummy vs our model):\n")
    out_eval.append_display_data(baseline_df)

    # LLM-audited flags on the metrics dict (e.g. high accuracy masking
    # weak recall on imbalanced data). Each flag has a severity and
    # explanation citing the actual numbers.
    flags = call_silent(flag_anomalies, metrics)
    out_eval.append_stdout("\nAnomaly flags (LLM audit):\n")
    out_eval.append_display_data(pd.DataFrame(flags))

    set_status("Evaluation finished")
    set_btn_done(btn_eval)


# Fix duplicate handlers

btn_eda._click_handlers.callbacks.clear()
btn_prep._click_handlers.callbacks.clear()
btn_model._click_handlers.callbacks.clear()
btn_eval._click_handlers.callbacks.clear()

btn_eda.on_click(on_eda)
btn_prep.on_click(on_prep)
btn_model.on_click(on_model)
btn_eval.on_click(on_eval)